# Notebook 02 — Simple Token Subgroup Baseline

This notebook trains the **simple token conditioning** model.

The subgroup is inserted directly into the RoBERTa input sequence as plain text:

```text
### COMMENT TO CLASSIFY
<comment>

### ANNOTATOR IDEOLOGY / IDENTITY
<subgroup>
```

Architecture:

```text
comment + subgroup text
        ↓
RoBERTa-base
        ↓
CLS representation
        ↓
linear classifier
        ↓
3-class probability distribution
```

No subgroup embedding.  
No FiLM.  
No retrieved context.

The notebook is controlled by `DISCOURSE`, so the same code can later be reused for `immigration` or `women`.


## 1. Imports and Configuration

In [2]:
import os
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    confusion_matrix,
    classification_report,
)
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

# ---------------------------------------------------------------------
# Main switch. Change this to "women" for the women-targeted discourse.
# ---------------------------------------------------------------------
DISCOURSE = "immigration"  # options: "immigration", "women"

DISCOURSE_CONFIG = {
    "immigration": {
        "processed_path": "immigration_processed.parquet",
        "subgroup_header": "ANNOTATOR IDEOLOGY",
        "allowed_subgroups": ["liberal", "conservative","extremely_liberal", "extremely_conservative", "slightly_liberal", "slightly_conservative", "no_opinion", "neutral"],
        "output_dir": "immigration_simple_token_outputs",
        "output_prefix": "immigration_simple_token",
    },
    "women": {
        "processed_path": "women_processed.parquet",
        "subgroup_header": "ANNOTATOR GENDER IDENTITY",
        # Leave as None to use every subgroup present in the processed file.
        "allowed_subgroups": None,
        "output_dir": "women_simple_token_outputs",
        "output_prefix": "women_simple_token",
    },
}

assert DISCOURSE in DISCOURSE_CONFIG, f"Unknown discourse: {DISCOURSE}"
CFG = DISCOURSE_CONFIG[DISCOURSE]

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

EXPERIMENT = DISCOURSE
OUTPUT_PREFIX = CFG["output_prefix"]

print("Discourse:", DISCOURSE)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR.resolve())


Discourse: immigration
Device: cuda
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/immigration_simple_token_outputs


In [3]:
def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Processed Data

This expects Notebook 1 to have created the processed file selected by `DISCOURSE`:

- `immigration_processed.parquet`
- `women_processed.parquet`


In [4]:
data_path = DATA_DIR / CFG["processed_path"]

if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path}. Make sure Notebook 1 created the processed file."
    )

comment_df = pd.read_parquet(data_path)

print("Loaded:", data_path)
print("Shape:", comment_df.shape)
display(comment_df.head(2))


Loaded: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data/immigration_processed.parquet
Shape: (3799, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,7,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,test,2,"[1.0, 0.0, 1.0]","[0.5, 0.0, 0.5]",1.000000,0,1.000000,"{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': 1.0, 'liberal': 1.0, 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, 'unknown': None}","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': [1.0, 0.0, 0.0], 'liberal': [0.0, 0.0, 1.0], 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, '...","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': [1.0, 0.0, 0.0], 'liberal': [0.0, 0.0, 1.0], 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, '..."
1,10,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...",train,3,"[1.0, 0.0, 2.0]","[0.3333333432674408, 0.0, 0.6666666865348816]",0.918296,2,1.333333,"{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': 1.0, 'neutral': 1.0, 'no_opinion': None, 'slightly_conservative': 1.0, 'slightly_liberal': None, 'unknown': None}","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': [0.0, 0.0, 1.0], 'neutral': [0.0, 0.0, 1.0], 'no_opinion': None, 'slightly_conservative': [1.0, 0.0, 0.0], 'slightly_libera...","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': [0.0, 0.0, 1.0], 'neutral': [0.0, 0.0, 1.0], 'no_opinion': None, 'slightly_conservative': [1.0, 0.0, 0.0], 'slightly_libera..."


In [5]:
required_columns = {
    "comment_id",
    "text",
    "split",
    "subgroup_distributions",
    "subgroup_counts",
}

missing = required_columns - set(comment_df.columns)
assert not missing, f"Dataset is missing columns: {missing}"

print("Required columns are present.")
print("Split counts:")
display(comment_df["split"].value_counts())


Required columns are present.
Split counts:


split
train         2659
test           590
validation     550
Name: count, dtype: int64

## 3. Helper Functions

In [6]:
def ensure_dict(value):
    """Return a dictionary whether the stored value is already a dict or a stringified dict."""
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    """Check that a distribution has the right length and sums approximately to one."""
    if len(dist) != num_labels:
        return False
    if any(float(p) < -tolerance for p in dist):
        return False
    return abs(sum(float(p) for p in dist) - 1.0) < tolerance


## 4. Expand Comment-Level Data into Subgroup-Level Examples

Notebook 1 gives one row per comment.

Notebook 2 needs one row per **comment-subgroup pair**.


In [7]:
def build_simple_token_input(text: str, subgroup: str, subgroup_header: str) -> str:
    """Build the simple token conditioning input string.

    The subgroup is plain text inside the RoBERTa sequence.
    There is no context and no separate subgroup embedding.
    """
    return (f"### COMMENT TO CLASSIFY{text}"f"### {subgroup_header}{subgroup}")


In [8]:
def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    experiment_name: str,
    subgroup_header: str,
    allowed_subgroups: list[str] | None = None,
) -> pd.DataFrame:

    rows = []
    skipped_none = 0
    skipped_invalid = 0
    skipped_not_allowed = 0

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():

            if allowed_subgroups is not None and subgroup not in allowed_subgroups:
                skipped_not_allowed += 1
                continue

            if target_distribution is None:
                skipped_none += 1
                continue

            target_distribution = [float(x) for x in target_distribution]

            if not is_valid_distribution(target_distribution):
                skipped_invalid += 1
                continue

            rows.append({
                "experiment": experiment_name,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "input_text": build_simple_token_input(row["text"], subgroup, subgroup_header),
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
                "target_entropy": float(entropy(target_distribution, base=2)),
            })

    expanded_df = pd.DataFrame(rows)

    print(f"{experiment_name}: skipped {skipped_none} None distributions")
    print(f"{experiment_name}: skipped {skipped_invalid} invalid distributions")
    print(f"{experiment_name}: skipped {skipped_not_allowed} not-allowed subgroups")
    print(f"{experiment_name}: created {len(expanded_df):,} subgroup examples")

    return expanded_df


In [9]:
model_examples = expand_subgroup_examples(
    comment_df=comment_df,
    experiment_name=DISCOURSE,
    subgroup_header=CFG["subgroup_header"],
    allowed_subgroups=CFG["allowed_subgroups"],
)

print("Subgroup examples:", model_examples.shape)
display(model_examples.head())

print("Example input text:")
print(model_examples.iloc[0]["input_text"][:1000])


immigration: skipped 21302 None distributions
immigration: skipped 0 invalid distributions
immigration: skipped 3799 not-allowed subgroups
immigration: created 9,090 subgroup examples
Subgroup examples: (9090, 11)


,experiment,comment_id,split,subgroup,subgroup_count,text,input_text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go...,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go...,"[0.0, 0.0, 1.0]",2,2.0,0.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","### COMMENT TO CLASSIFYThey'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we'...","[0.0, 0.0, 1.0]",2,2.0,0.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","### COMMENT TO CLASSIFYThey'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we'...","[0.0, 0.0, 1.0]",2,2.0,0.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","### COMMENT TO CLASSIFYThey'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we'...","[1.0, 0.0, 0.0]",0,0.0,0.0


Example input text:
### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill dig him the hole to get there myself### ANNOTATOR IDEOLOGYextremely_liberal


In [10]:
print("Split × subgroup counts")
display(pd.crosstab(model_examples["split"], model_examples["subgroup"]))


Split × subgroup counts


subgroup,conservative,extremely_conservative,extremely_liberal,liberal,neutral,no_opinion,slightly_conservative,slightly_liberal
split,,,,,,,,
test,163,46,197,308,257,46,154,213
train,739,252,907,1407,1027,206,756,1066
validation,157,65,181,311,228,37,169,198


## 5. Select Model Data

In [11]:
model_df = model_examples.copy()

print(model_df.shape)
display(model_df.head())


(9090, 11)


,experiment,comment_id,split,subgroup,subgroup_count,text,input_text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go...,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go...,"[0.0, 0.0, 1.0]",2,2.0,0.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","### COMMENT TO CLASSIFYThey'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we'...","[0.0, 0.0, 1.0]",2,2.0,0.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","### COMMENT TO CLASSIFYThey'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we'...","[0.0, 0.0, 1.0]",2,2.0,0.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","### COMMENT TO CLASSIFYThey'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we'...","[1.0, 0.0, 0.0]",0,0.0,0.0


In [12]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0, "Training set is empty."
assert len(val_df) > 0, "Validation set is empty."
assert len(test_df) > 0, "Test set is empty."


Train: (6360, 11)
Val: (1346, 11)
Test: (1384, 11)


## 6. Tokenization and Dataset Class

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [14]:
class HateSpeechDistributionDataset(Dataset):
    """PyTorch dataset for subgroup-conditioned distribution prediction."""

    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 192):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "target_distribution": torch.tensor(row["target_distribution"], dtype=torch.float),
        }


In [15]:
train_dataset = HateSpeechDistributionDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = HateSpeechDistributionDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = HateSpeechDistributionDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([16, 192]),
 'attention_mask': torch.Size([16, 192]),
 'target_distribution': torch.Size([16, 3])}

## 7. Model

The model returns log-probabilities because `nn.KLDivLoss` expects log-probabilities as input.


In [16]:
class SimpleTokenRoBERTa(nn.Module):
    """RoBERTa distribution predictor with subgroup represented as plain input text."""

    def __init__(self, model_name: str, num_labels: int = 3, dropout: float = 0.1):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_embedding))

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


In [17]:
model = SimpleTokenRoBERTa(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)
print("RoBERTa layers:", model.encoder.config.num_hidden_layers)
print("Hidden size:", model.encoder.config.hidden_size)
print("Attention heads:", model.encoder.config.num_attention_heads)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 3980
Warmup steps: 398
RoBERTa layers: 12
Hidden size: 768
Attention heads: 12


## 8. Metric Functions

In [18]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """KL(y_true || y_pred) per row."""
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Jensen-Shannon divergence per row."""
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Cross-entropy H(y_true, y_pred) per row."""
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro")),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 9. Training and Evaluation Functions

In [19]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs["log_probs"], targets)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 10. Train Baseline

In [20]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)


Epoch 1/10
Train KL: 0.7144 | Val KL: 0.5989
Train JS: 0.2758 | Val JS: 0.2360
Val Acc: 0.7786 | Val Macro F1: 0.4627
Saved new best model to immigration_simple_token_outputs/immigration_simple_token_best_model.pt
Epoch 2/10
Train KL: 0.6103 | Val KL: 0.6095
Train JS: 0.2269 | Val JS: 0.2146
Val Acc: 0.7689 | Val Macro F1: 0.4632
Epoch 3/10
Train KL: 0.5531 | Val KL: 0.6115
Train JS: 0.2044 | Val JS: 0.2098
Val Acc: 0.7741 | Val Macro F1: 0.4666
Epoch 4/10
Train KL: 0.5094 | Val KL: 0.6554
Train JS: 0.1882 | Val JS: 0.2082
Val Acc: 0.7779 | Val Macro F1: 0.4613
Epoch 5/10
Train KL: 0.4720 | Val KL: 0.6526
Train JS: 0.1753 | Val JS: 0.2135
Val Acc: 0.7697 | Val Macro F1: 0.4609
Epoch 6/10
Train KL: 0.4333 | Val KL: 0.6719
Train JS: 0.1630 | Val JS: 0.2165
Val Acc: 0.7689 | Val Macro F1: 0.4590
Epoch 7/10
Train KL: 0.3963 | Val KL: 0.7958
Train JS: 0.1534 | Val JS: 0.2085
Val Acc: 0.7801 | Val Macro F1: 0.4687
Epoch 8/10
Train KL: 0.3742 | Val KL: 0.7714
Train JS: 0.1462 | Val JS: 0.2157

,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.714379,0.275768,0.746830,0.740094,0.369744,0.625072,0.068073,0.063928,0.714379,0.598911,0.236033,0.633089,0.778603,0.462709,0.537542,0.171520,0.177057,0.598911
1,2,0.610325,0.226932,0.642776,0.758962,0.450354,0.497381,0.137100,0.133504,0.610325,0.609538,0.214641,0.643716,0.768945,0.463165,0.450448,0.169735,0.161345,0.609538
2,3,0.553080,0.204385,0.585531,0.785220,0.484519,0.432578,0.135146,0.122492,0.553080,0.611456,0.209802,0.645634,0.774146,0.466575,0.444969,0.140740,0.129933,0.611456
3,4,0.509400,0.188176,0.541851,0.795440,0.496974,0.391952,0.156483,0.144175,0.509400,0.655366,0.208211,0.689544,0.777860,0.461294,0.430139,0.171779,0.161711,0.655366
4,5,0.471966,0.175334,0.504417,0.808962,0.516906,0.362016,0.157242,0.143878,0.471966,0.652562,0.213460,0.686741,0.769688,0.460949,0.445509,0.153795,0.134463,0.652562
5,6,0.433308,0.162994,0.465759,0.811321,0.537312,0.335971,0.155652,0.146342,0.433308,0.671924,0.216478,0.706102,0.768945,0.459044,0.445023,0.156437,0.136655,0.671924
6,7,0.396311,0.153431,0.428762,0.821069,0.580000,0.320159,0.175213,0.163556,0.396311,0.795841,0.208508,0.830019,0.780089,0.468717,0.411107,0.179448,0.158848,0.795841
7,8,0.374205,0.146186,0.406656,0.819025,0.603456,0.308307,0.177426,0.169592,0.374205,0.771386,0.215670,0.805564,0.772660,0.487204,0.431188,0.152924,0.138065,0.771386
8,9,0.354398,0.140108,0.386849,0.823742,0.627740,0.299863,0.192818,0.182198,0.354398,0.838301,0.210114,0.872479,0.763744,0.460422,0.415237,0.179123,0.159403,0.838301
9,10,0.330617,0.133234,0.363068,0.834119,0.659348,0.292205,0.195661,0.184204,0.330617,0.861065,0.213898,0.895243,0.762259,0.466073,0.416037,0.176039,0.155818,0.861065


In [21]:
history_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Saved: immigration_simple_token_outputs/immigration_simple_token_training_history.csv


## 11. Test Evaluation

In [22]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

test_metrics


/tmp/ipykernel_7716/3949287068.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.607476532459259,
 'js_mean': 0.24174678687651296,
 'cross_entropy_mean': 0.6319013833999634,
 'accuracy': 0.7651734104046243,
 'macro_f1': 0.4469731578000194,
 'expected_label_mae': 0.5574516446438667,
 'entropy_pearson': 0.058430098330426394,
 'entropy_spearman': 0.06230167918018363,
 'loss': 0.6074765576103519}

In [23]:
metrics_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


Saved: immigration_simple_token_outputs/immigration_simple_token_test_metrics.json


## 12. Save Test Predictions

In [24]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())


,experiment,comment_id,split,subgroup,subgroup_count,text,input_text,target_distribution,target_majority_label,target_expected_label,target_entropy,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go...,"[1.0, 0.0, 0.0]",0,0.0,0.0,"[1.0, 0.0, 0.0]","[0.39488596, 0.09864707, 0.506467]",2,1.111581,1.356056,0.929158,0.400454,0.929158
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,### COMMENT TO CLASSIFY\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go...,"[0.0, 0.0, 1.0]",2,2.0,0.0,"[0.0, 0.0, 1.0]","[0.40092462, 0.0999659, 0.49910942]",2,1.098185,1.361180,0.694930,0.311984,0.694930
2,immigration,66,test,extremely_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,### COMMENT TO CLASSIFYGet ready to be flogged. India is ready to flog all cow dung eating shameless fucktards### ANNOTATOR IDEOLOGYextremely_liberal,"[0.0, 0.0, 1.0]",2,2.0,0.0,"[0.0, 0.0, 1.0]","[0.2637468, 0.06883403, 0.6674192]",2,1.403672,1.162204,0.404337,0.190377,0.404337
3,immigration,66,test,slightly_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,### COMMENT TO CLASSIFYGet ready to be flogged. India is ready to flog all cow dung eating shameless fucktards### ANNOTATOR IDEOLOGYslightly_liberal,"[1.0, 0.0, 0.0]",0,0.0,0.0,"[1.0, 0.0, 0.0]","[0.27375543, 0.072752886, 0.6534917]",2,1.379736,1.187814,1.295520,0.521843,1.295520
4,immigration,108,test,conservative,1,"I am secretly hoping he is simply saying that to deter them from coming, he keeps suspending the Haiti deportations but look what happened when he threatened to do it the first time, we got flooded with those lazy fu...","### COMMENT TO CLASSIFYI am secretly hoping he is simply saying that to deter them from coming, he keeps suspending the Haiti deportations but look what happened when he threatened to do it the first time, we got flo...","[0.0, 0.0, 1.0]",2,2.0,0.0,"[0.0, 0.0, 1.0]","[0.4435514, 0.09354497, 0.46290356]",2,1.019352,1.354350,0.770236,0.341361,0.770237


In [25]:
predictions_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print("Saved:", predictions_path)


predictions_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_predictions.csv"
predictions_df.to_csv(predictions_csv_path, index=False)
print("Saved:", predictions_csv_path)


Saved: immigration_simple_token_outputs/immigration_simple_token_test_predictions.parquet
Saved: immigration_simple_token_outputs/immigration_simple_token_test_predictions.csv


## 13. Basic Error Analysis

In [26]:
print("Worst examples by KL divergence")

display(
    predictions_df
    .sort_values("kl", ascending=False)
    [[
        "comment_id",
        "subgroup",
        "text",
        "target_distribution",
        "pred_distribution",
        "kl",
        "js",
    ]]
    .head(10)
)


Worst examples by KL divergence


,comment_id,subgroup,text,target_distribution,pred_distribution,kl,js
1088,44261,liberal,Caucasian and Thai,"[0.0, 1.0, 0.0]","[0.9174585, 0.046790008, 0.035751514]",3.062086,0.862119
61,4177,neutral,Down with Mexicans,"[0.0, 1.0, 0.0]","[0.9048531, 0.047816377, 0.047330473]",3.040387,0.859826
319,17934,slightly_conservative,Want benefits of being a citizen take citizenship?,"[0.0, 1.0, 0.0]","[0.9108788, 0.049813736, 0.039307542]",2.999465,0.855407
911,41510,conservative,It's getting worse everywhere refugees get priority over citizens,"[0.0, 1.0, 0.0]","[0.8887459, 0.05198948, 0.05926459]",2.956714,0.850655
1018,43256,neutral,Can we just nuke Mexico?,"[0.0, 0.0, 1.0]","[0.88470227, 0.055258177, 0.06003962]",2.812751,0.833597
1182,46045,liberal,We need to bomb Mexico into rubble.,"[0.0, 1.0, 0.0]","[0.83797926, 0.060144708, 0.10187598]",2.811002,0.833379
1235,47255,slightly_liberal,i am really starting to see why people in other countries (i.e. Europe) hate Americans...,"[0.0, 0.0, 1.0]","[0.8731553, 0.06346446, 0.06338026]",2.758603,0.826741
681,33782,slightly_liberal,One deportation. I was a promised a million.,"[0.0, 1.0, 0.0]","[0.85589725, 0.06438991, 0.0797129]",2.742798,0.824692
1279,48123,slightly_conservative,They should be dumped 300 miles offshore if they make it back alive then send them back to Mexico in a body bag,"[0.0, 1.0, 0.0]","[0.8059811, 0.06475531, 0.12926354]",2.737139,0.823953
853,40431,neutral,"MAKE SURE THAT ALL AMERICANS GET OUT OF OTHER PEOPLES' LANDS INCLUDING THE NORTH, SOUTH AND CENTRAL AMERICAS !!!!","[0.0, 0.0, 1.0]","[0.87237465, 0.0593128, 0.068312526]",2.683662,0.816834


In [27]:
print("Performance by subgroup")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")

display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)

print("Saved:", subgroup_metrics_path)


Performance by subgroup


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,extremely_conservative,46,0.472854,0.201671,0.483399,0.869565,0.556306,0.508777,0.070167,0.061758
0,conservative,163,0.500304,0.204765,0.512236,0.834356,0.511037,0.475696,0.021685,0.027767
5,no_opinion,46,0.566726,0.225553,0.585646,0.847826,0.557890,0.519581,-0.170354,-0.102090
6,slightly_conservative,154,0.577238,0.233096,0.588798,0.759740,0.424238,0.549976,-0.054434,-0.050474
3,liberal,308,0.588830,0.234402,0.629092,0.775974,0.453372,0.536117,0.120573,0.137246
2,extremely_liberal,197,0.639998,0.254256,0.660715,0.730964,0.430749,0.584806,0.020386,0.028349
4,neutral,257,0.667239,0.260413,0.700662,0.747082,0.429648,0.598475,0.080374,0.079316
7,slightly_liberal,213,0.674004,0.264983,0.691149,0.713615,0.392172,0.600164,0.058386,0.056377


Saved: immigration_simple_token_outputs/immigration_simple_token_subgroup_metrics.csv


from sklearn.metrics import confusion_matrix, classification_report

true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\nAverage target distribution:")
print(np.vstack(predictions_df["true_distribution"].to_numpy()).mean(axis=0))

print("\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))


## 14. Notes for Unified Evaluation

This simple token notebook saves standardized files into `OUTPUT_DIR`:

- `*_best_model.pt`
- `*_training_history.csv`
- `*_test_metrics.json`
- `*_test_predictions.parquet`
- `*_test_predictions.csv`
- `*_subgroup_metrics.csv`

These can be loaded by the unified evaluation notebook and compared against Original A, Embedding, FiLM, and context variants.
